<a href="https://colab.research.google.com/github/RicketyMajor/health-access-ds/blob/main/notebooks/01_eda_censo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geopandas matplotlib contextily pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.7 MB/s eta 0:00:00


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

# 1. Cargar el dato crudo (Ajusta el nombre del archivo al que descargues del INE)
# Esto puede tardar unos segundos porque los shapefiles son pesados.
ruta_shp = "Manzanas_Censo_2017_RM.shp"
gdf_manzanas = gpd.read_file(ruta_shp)

# 2. Exploración empírica: Ver qué columnas tenemos
print("Columnas disponibles:", gdf_manzanas.columns.tolist())

# Supongamos que la columna de personas mayores de 65 años se llama 'EDAD_65YMAS'
# (Debes verificar el nombre exacto en el diccionario de datos del INE)
# Y filtramos solo las manzanas que tengan población de tercera edad.
gdf_tercera_edad = gdf_manzanas[gdf_manzanas['EDAD_65YMAS'] > 0].copy()

# 3. Mejora aplicada: Guardar como GeoParquet para que tus compañeros lo lean en 1 segundo
ruta_parquet = "manzanas_tercera_edad.parquet"
gdf_tercera_edad.to_parquet(ruta_parquet)
print(f"Datos limpios guardados exitosamente en formato Parquet.")

# 4. Prueba visual: Mapear una comuna específica (Ej: Puente Alto o Maipú)
# Filtramos por el código de comuna para no saturar la RAM renderizando todo Santiago
gdf_comuna = gdf_tercera_edad[gdf_tercera_edad['NOM_COMUNA'] == 'PUENTE ALTO']

# Aseguramos que el sistema de coordenadas (CRS) sea Web Mercator para el mapa base
gdf_comuna = gdf_comuna.to_crs(epsg=3857)

# Graficar
fig, ax = plt.subplots(figsize=(12, 12))
# El color (cmap) indicará la concentración de adultos mayores
gdf_comuna.plot(column='EDAD_65YMAS', cmap='OrRd', legend=True, ax=ax, alpha=0.7)

# Añadir el mapa base de calles reales de fondo
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.title('Concentración de Población > 65 años por Manzana en Puente Alto')
plt.show()